# Checkpoint point-in-time PhoBERT (đánh giá ngoài mẫu)

Notebook này tạo checkpoint đóng băng chỉ từ nhãn công bố **nghiêm ngặt trước 2024-10-21** — ngày quan sát kiểm thử đầu tiên của thang dự báo — để kết quả forecast là đánh giá ngoài mẫu hợp lệ thay vì hồi cứu.

Trên Kaggle: bật **GPU** và **Internet**, đính kèm Dataset `phuocthoai/stock-trend-forecasting` chứa đúng một tệp `labeled_merged.csv`, rồi chạy từ trên xuống dưới. Ô kiểm tra ở cuối phải in `training_size: 1049` và `label_cutoff: 2024-10-21`. Khi xong, chọn **Save Version → Save & Run All** để ZIP xuất hiện ở tab Output.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nphuoctho/stock-trend-forecasting.git'
BRANCH = 'feat/point-in-time-checkpoint'
REPO_DIR = Path('/kaggle/working/stock-trend-forecasting') if Path('/kaggle').exists() else Path('/content/stock-trend-forecasting')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', '--detach', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = {**os.environ, 'PYTHONPATH': os.pathsep.join(part for part in (str(SRC_DIR), os.environ.get('PYTHONPATH', '')) if part)}

# Giữ torch CUDA của Kaggle. Cặp này đã được ghi trong artifact CV thành công.
if importlib.util.find_spec('torchvision') is not None:
    check = subprocess.run([sys.executable, '-c', 'import torchvision'], capture_output=True, text=True)
    if check.returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'timm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers==5.15.1', 'tokenizers>=0.22,<=0.23.0', 'huggingface-hub>=1.5,<2.0', 'accelerate>=1.1,<2', 'sentencepiece>=0.2,<1'], check=True)
importlib.invalidate_caches()
subprocess.run([sys.executable, '-c', "import torch; assert torch.__version__ == '2.10.0+cu128', f'PyTorch {torch.__version__}; cần phiên Kaggle mới với 2.10.0+cu128.'"], check=True, env=SUBPROCESS_ENV)
subprocess.run([sys.executable, '-c', "from transformers import Trainer, TrainingArguments; TrainingArguments(output_dir='/tmp/stf-check', eval_strategy='no', save_strategy='no', use_cpu=True)"], check=True, env=SUBPROCESS_ENV)
print('Repository ready:', REPO_DIR)

In [ ]:
import torch
import pandas as pd
from stf.sentiment.dataset import file_fingerprint, load_labeled

if not torch.cuda.is_available():
    raise RuntimeError('GPU chưa được bật. Chọn T4 GPU rồi chạy lại notebook.')
INPUT_ROOT = Path('/kaggle/input')
matches = sorted(path for path in INPUT_ROOT.rglob('labeled_merged.csv') if path.is_file())
if len(matches) != 1:
    found = '\n'.join(str(path) for path in matches) or '(không có tệp nào)'
    raise FileNotFoundError(
        'Cần đúng một tệp labeled_merged.csv trong Kaggle Input. Tìm thấy:\n' + found
    )
DATA_PATH = matches[0]
print('Using attached label file:', DATA_PATH)
raw = pd.read_csv(DATA_PATH)
assert len(raw) == 1306, f'Cần 1.306 nhãn đã hợp nhất, nhận {len(raw)}.'
assert raw['annotation_source'].eq('human_reviewed').all()
assert raw['annotation_status'].eq('REVIEWED').all()
reviewed = load_labeled(DATA_PATH)
print('GPU:', torch.cuda.get_device_name(0))
print('Rows:', len(reviewed))
print('SHA-256:', file_fingerprint(DATA_PATH))
print(reviewed['label'].value_counts().to_string())

In [ ]:
INPUT_VARIANT = 'title_context'
TRUNCATION_STRATEGY = 'head_tail'
CLASS_WEIGHTING = 'inverse_frequency'
EPOCHS = 5
BATCH_SIZE = 16
SEED = 42
CV_RESULTS = REPO_DIR / 'outputs/sentiment-cv-merged/cv_results.json'
OUTPUT_DIR = Path('/kaggle/working/point-in-time') if Path('/kaggle').exists() else Path('/content/point-in-time')

command = [
    sys.executable, '-m', 'stf.cli', 'sentiment-refit',
    '--data', str(DATA_PATH),
    '--cv-results', str(CV_RESULTS),
    '--input-variant', INPUT_VARIANT,
    '--truncation-strategy', TRUNCATION_STRATEGY,
    '--class-weighting', CLASS_WEIGHTING,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--seed', str(SEED),
    '--before-date', '2024-10-21',
    '--output', str(OUTPUT_DIR),
]
print(' '.join(command))
subprocess.run(command, check=True, env=SUBPROCESS_ENV)

In [ ]:
import json

EXPECTED_CUTOFF = '2024-10-21'
EXPECTED_SIZE = 1049

manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))
provenance = manifest['provenance']

assert manifest['run_type'] == 'full_data_refit'
assert manifest['training_size'] == EXPECTED_SIZE, manifest['training_size']
assert provenance['label_cutoff'] == EXPECTED_CUTOFF, provenance['label_cutoff']
assert provenance['source_file_sha256'] == file_fingerprint(DATA_PATH)
assert 'test_metrics' not in manifest
assert (OUTPUT_DIR / 'best' / 'config.json').is_file()

print('training_size:', manifest['training_size'])
print('label_cutoff:', provenance['label_cutoff'])
print('class_distribution:', manifest['class_distribution'])
print(json.dumps(manifest['selection'], ensure_ascii=False, indent=2))
print('Checkpoint:', OUTPUT_DIR / 'best')

In [ ]:
import shutil

archive_base = Path.cwd() / OUTPUT_DIR.name
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
print('Archive:', archive_path)
print(f'Size: {archive_path.stat().st_size / 1024**2:.1f} MiB')
print('Kaggle: chọn Save Version → Save & Run All, rồi tải ZIP ở tab Output của phiên bản đã lưu.')